# OMNI MOVIE STUDIO — FREE GPU WORKER v5 (Colab T4)
Drive-bridge mode: the agent queues shot jobs in Google Drive (omni-movie/queue/), this notebook renders them on the free GPU and writes clips back (omni-movie/results/). No public URLs needed.
Runtime → Run all. Wait for **WORKER READY**, then **keep this tab open** — the worker loop keeps rendering until the agent's queue is empty. Models are cached in this runtime, so re-runs are fast.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
import os
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/omni-movie'
os.makedirs(OUT, exist_ok=True)
print('drive ready:', OUT)

In [ ]:
# 1) ComfyUI (starts FIRST, downloads models next cell)
import os
if not os.path.exists('/content/ComfyUI/.git'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI > /dev/null 2>&1
%pip -q install -r /content/ComfyUI/requirements.txt
%cd /content/ComfyUI
import subprocess, threading, time, urllib.request
threading.Thread(target=lambda: subprocess.run(['python','main.py','--port','8188','--listen','0.0.0.0','--dont-print-server']), daemon=True).start()
print('comfyui thread started (models download in next cell...)')

In [ ]:
# 2) MODELS — SDXL (stills) + Wan 2.1 I2V 480p fp8 (animation). Free open weights, ~18GB.
M='https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files'
!wget -q -nc -P /content/ComfyUI/models/unet {M}/diffusion_models/wan2.1_i2v_480p_9.5B_fp8_e4m3fn.safetensors
!wget -q -nc -P /content/ComfyUI/models/text_encoders {M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
!wget -q -nc -P /content/ComfyUI/models/clip_vision {M}/clip_vision/clip_vision_h.safetensors
!wget -q -nc -P /content/ComfyUI/models/vae {M}/vae/wan_2.1_vae.safetensors
!wget -q -nc -P /content/ComfyUI/models/checkpoints https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors
import subprocess
for d in ['unet','text_encoders','clip_vision','vae','checkpoints']:
    print(d, ':', subprocess.run(['ls','-lh',f'/content/ComfyUI/models/{d}'],capture_output=True,text=True).stdout.strip().split('\n')[-1])
print('MODELS DOWNLOADED')

In [ ]:
# 3) Wav2Lip — free lip-sync
!git clone --depth 1 https://github.com/Rudrabha/Wav2Lip /content/Wav2Lip > /dev/null 2>&1
%cd /content/Wav2Lip
!mkdir -p checkpoints face_detection/detection/sfd
!wget -q https://github.com/justinjohn0306/Wav2Lip/releases/download/Models/wav2lip_gan.pth -O checkpoints/wav2lip_gan.pth || echo 'wav2lip weights: get from Wav2Lip repo releases'
!wget -q https://github.com/justinjohn0306/Wav2Lip/releases/download/Models/s3fd.pth -O face_detection/detection/sfd/s3fd.pth || echo 's3fd: get from repo'
%pip -q install librosa==0.10.1 numba==0.58.1
print('wav2lip ready (env: LIPSYNC_ENGINE=wav2lip)')

In [ ]:
# 4) TTS — Coqui XTTS v2 via the maintained 'coqui-tts' fork (works on Python 3.13).
# If that still fails, edge-tts (free Microsoft neural Hindi voices) takes over — same /tts API.
import importlib, subprocess, sys

def try_import(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

if not try_import('TTS'):
    print('installing coqui-tts (py3.13 fork)...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'coqui-tts'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('coqui-tts install failed:', r.stderr[-400:])
if try_import('TTS'):
    BACKEND = 'xtts'
else:
    print('XTTS unavailable -> installing edge-tts fallback')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'edge-tts'], capture_output=True)
    BACKEND = 'edge'
print('TTS backend:', BACKEND)

server = '''
import json
from http.server import BaseHTTPRequestHandler, HTTPServer

BACKEND = "BACKEND_PLACEHOLDER"
HINDI_VOICES = ["hi-IN-MadhurNeural", "hi-IN-SwaraNeural", "hi-IN-HemaNeural"]
SPEAKERS = ["Claribel Dervla", "Damien Black", "Viktor Menelaos", "Daisy Studious", "Alma Mar\u00eda"]
XTTS = None

def pick_voice(name):
    return HINDI_VOICES[hash(name) % len(HINDI_VOICES)]

def pick_speaker(name):
    return SPEAKERS[hash(name) % len(SPEAKERS)]

if BACKEND == "xtts":
    from TTS.api import TTS
    XTTS = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
    print("XTTS v2 model loaded")

class H(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/health":
            self.send_response(200); self.end_headers(); self.wfile.write(b"ok")
        else:
            self.send_response(404); self.end_headers()
    def do_POST(self):
        try:
            n = int(self.headers.get("content-length", 0)); body = json.loads(self.rfile.read(n))
            out = body.get("out", "/tmp/tts_out.wav")
            text = body.get("text", "")
            name = str(body.get("voice") or body.get("speaker_wav") or "narrator")
            if BACKEND == "xtts":
                XTTS.tts_to_file(text=text, language=body.get("language", "hi"),
                                 speaker=pick_speaker(name), file_path=out)
            else:
                import edge_tts, asyncio
                asyncio.run(edge_tts.Communicate(text, pick_voice(name)).save(out))
            data = open(out, "rb").read()
            self.send_response(200); self.send_header("content-type", "audio/wav")
            self.send_header("content-length", str(len(data))); self.end_headers()
            self.wfile.write(data)
        except Exception as e:
            import traceback; traceback.print_exc()
            self.send_response(500); self.end_headers(); self.wfile.write(str(e).encode())
HTTPServer(("0.0.0.0", 8020), H).serve_forever()
'''
server = server.replace("BACKEND_PLACEHOLDER", BACKEND)
open('/content/xtts_server.py', 'w').write(server)
import threading, subprocess
threading.Thread(target=lambda: subprocess.run(['python', '/content/xtts_server.py']), daemon=True).start()
print('tts server thread started (backend:', BACKEND, ')')

In [ ]:
# 6) RENDER WORKER — polls Drive for shot jobs, renders still->animation->voice->lip-sync, writes clips back to Drive.
import json, os, time, glob, shutil, random, traceback, subprocess
import requests, urllib.request

QUEUE = '/content/drive/MyDrive/omni-movie/queue'
DONE  = '/content/drive/MyDrive/omni-movie/processed'
RES   = '/content/drive/MyDrive/omni-movie/results'
for d in (QUEUE, DONE, RES):
    os.makedirs(d, exist_ok=True)

COMFY = 'http://127.0.0.1:8188'
TTS   = 'http://127.0.0.1:8020'
WAV2LIP_OK = os.path.exists('/content/Wav2Lip/checkpoints/wav2lip_gan.pth')
print('wav2lip available:', WAV2LIP_OK)

def wait_service(url, label, timeout=1200):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            urllib.request.urlopen(url, timeout=5)
            print(label, 'UP'); return True
        except Exception:
            time.sleep(10)
    print(label, 'NOT UP'); return False

comfy_ok = wait_service(COMFY + '/system_stats', 'comfyui')
tts_ok   = wait_service(TTS + '/health', 'tts')
print('WORKER READY — comfy:', comfy_ok, '| tts:', tts_ok, '| wav2lip:', WAV2LIP_OK)

def http_json(url, payload=None, timeout=30):
    if payload is None:
        return json.loads(urllib.request.urlopen(url, timeout=timeout).read())
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()

def comfy_queue(workflow, timeout=2400):
    pid = http_json(COMFY + '/prompt', {'prompt': workflow, 'client_id': 'omni-worker'})['prompt_id']
    t0 = time.time()
    while time.time() - t0 < timeout:
        time.sleep(3)
        try:
            h = http_json(COMFY + '/history/' + pid)
        except Exception:
            continue
        if pid not in h:
            continue
        entry = h[pid]
        for node in entry.get('outputs', {}).values():
            for im in node.get('images', []):
                if im.get('type') == 'output':
                    return im['filename'], im.get('subfolder', '')
            for g in node.get('gifs', []):
                if g.get('type') == 'output':
                    return g['filename'], g.get('subfolder', '')
        st = entry.get('status', {})
        if st.get('status_str') == 'error':
            raise RuntimeError('comfy error: ' + json.dumps(st)[:400])
    raise RuntimeError('comfy timeout')

def comfy_fetch(filename, subfolder, out):
    url = COMFY + '/view?filename=' + requests.utils.quote(filename) + '&subfolder=' + requests.utils.quote(subfolder or '') + '&type=output'
    urllib.request.urlretrieve(url, out)

def comfy_upload_image(path):
    with open(path, 'rb') as f:
        r = requests.post(COMFY + '/upload/image', files={'image': (os.path.basename(path), f)}, data={'overwrite': 'true'})
    r.raise_for_status()
    return r.json()['name']

def text2img(prompt, out_png, seed=None):
    wf = {
      '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'sd_xl_base_1.0.safetensors'}},
      '2': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1', 1]}},
      '3': {'class_type': 'CLIPTextEncode', 'inputs': {'text': 'blurry, low quality, watermark, text, deformed', 'clip': ['1', 1]}},
      '4': {'class_type': 'EmptyLatentImage', 'inputs': {'width': 1024, 'height': 576, 'batch_size': 1}},
      '5': {'class_type': 'KSampler', 'inputs': {'seed': seed if seed is not None else random.randint(1, 10**9),
              'steps': 25, 'cfg': 7, 'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0,
              'model': ['1', 0], 'positive': ['2', 0], 'negative': ['3', 0], 'latent_image': ['4', 0]}},
      '6': {'class_type': 'VAEDecode', 'inputs': {'samples': ['5', 0], 'vae': ['1', 2]}},
      '7': {'class_type': 'SaveImage', 'inputs': {'images': ['6', 0], 'filename_prefix': 'omni'}},
    }
    fn, sub = comfy_queue(wf)
    comfy_fetch(fn, sub, out_png)

def i2v(image_path, motion_prompt, seconds, out_mp4):
    frames = min(81, int(seconds) * 16 + 1)
    wf = {
      '1': {'class_type': 'WanImageToVideo', 'inputs': {'model': ['4', 0], 'conditioning': ['7', 0], 'vae': ['3', 0],
              'clip_vision_output': ['6', 0], 'width': 816, 'height': 480, 'length': frames, 'batch_size': 1}},
      '2': {'class_type': 'LoadImage', 'inputs': {'image': comfy_upload_image(image_path)}},
      '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'wan_2.1_vae.safetensors'}},
      '4': {'class_type': 'UNETLoader', 'inputs': {'unet_name': 'wan2.1_i2v_480p_9.5B_fp8_e4m3fn.safetensors', 'weight_dtype': 'fp8_e4m3fn'}},
      '5': {'class_type': 'CLIPVisionLoader', 'inputs': {'clip_name': 'clip_vision_h.safetensors'}},
      '6': {'class_type': 'CLIPVisionEncode', 'inputs': {'clip_vision': ['5', 0], 'image': ['2', 0], 'crop': 'center'}},
      '7': {'class_type': 'WanImageClipTextEncode', 'inputs': {'positive': motion_prompt,
              'negative': 'blurry, low quality, watermark, distorted face, static',
              'clip': ['8', 0], 'strength': 1, 'force_offload': True}},
      '8': {'class_type': 'CLIPLoader', 'inputs': {'type': 'wan', 'clip_name': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'device': 'default'}},
      '10': {'class_type': 'VAEDecode', 'inputs': {'samples': ['1', 0], 'vae': ['3', 0]}},
      '9': {'class_type': 'SaveVideo', 'inputs': {'filename_prefix': 'omni_i2v', 'crf': 20, 'pingpong': False, 'format': 'auto', 'images': ['10', 0]}},
    }
    fn, sub = comfy_queue(wf)
    comfy_fetch(fn, sub, out_mp4)

def tts_wav(text, character, out_wav):
    r = requests.post(TTS + '/tts', json={'text': text, 'language': 'hi', 'voice': character or 'narrator'}, timeout=600)
    r.raise_for_status()
    open(out_wav, 'wb').write(r.content)

def render_shot(job):
    sid = job['id']
    still = f'/content/shot-{sid}-still.png'
    anim  = f'/content/shot-{sid}-anim.mp4'
    voice = f'/content/shot-{sid}-voice.wav'
    out   = f'/content/shot-{sid}.mp4'

    print('  [still]', flush=True)
    text2img(job['imagePrompt'], still)
    print('  [i2v]', job.get('seconds', 4), 's', flush=True)
    i2v(still, job['motionPrompt'], job.get('seconds', 4), anim)

    if job.get('line'):
        print('  [tts]', job.get('character'), flush=True)
        tts_wav(job['line'], job.get('character'), voice)
        lip_done = False
        if WAV2LIP_OK:
            try:
                print('  [wav2lip]', flush=True)
                subprocess.run(['python', '/content/Wav2Lip/inference.py', '--checkpoint_path',
                                '/content/Wav2Lip/checkpoints/wav2lip_gan.pth', '--face', anim,
                                '--audio', voice, '--outfile', f'/content/shot-{sid}-lip'], check=True, timeout=900)
                cand = glob.glob(f'/content/shot-{sid}-lip/**/*.mp4', recursive=True) + glob.glob(f'/content/shot-{sid}-lip*.mp4')
                if cand:
                    shutil.move(cand[0], out); lip_done = True
            except Exception as e:
                print('  wav2lip failed -> voice-over fallback:', str(e)[:150], flush=True)
        if not lip_done:
            subprocess.run(['ffmpeg', '-y', '-v', 'error', '-stream_loop', '-1', '-i', anim, '-i', voice,
                            '-map', '0:v', '-map', '1:a', '-c:v', 'copy', '-c:a', 'aac', '-shortest', out], check=True)
    else:
        shutil.copy(anim, out)

    dest = os.path.join(RES, f'shot-{sid}.mp4')
    shutil.move(out, dest)
    print('  uploaded to Drive:', dest, flush=True)

while True:
    jobs = sorted(glob.glob(QUEUE + '/*.json'))
    if not jobs:
        time.sleep(20)
        continue
    for jf in jobs:
        try:
            job = json.load(open(jf))
            print('rendering shot', job['id'], f"({job.get('kind')})", flush=True)
            render_shot(job)
            status = {'id': job['id'], 'status': 'ok'}
        except Exception as e:
            traceback.print_exc()
            status = {'id': (job.get('id') if isinstance(job, dict) else None), 'status': 'failed', 'error': str(e)[:300]}
        shutil.move(jf, os.path.join(DONE, os.path.basename(jf)))
        with open(os.path.join(DONE, f"status-{status['id']}.json"), 'w') as f:
            json.dump(status, f)
        time.sleep(5)
    time.sleep(10)